## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [7]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os


In [8]:
MODEL = "llama-3.1-8b-instant"
DB_NAME = "vector_db"
load_dotenv(override=True)

python-dotenv could not parse statement starting at line 3


True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [9]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [10]:
retriever = vectorstore.as_retriever()


# Chỉnh ChatOpenAI để trỏ về máy chủ của Groq
llm = ChatOpenAI(
    temperature=0, 
    model_name=MODEL,
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

### These LangChain objects implement the method `invoke()`

In [11]:
retriever.invoke("Who is Avery?")

[Document(id='e3910ba2-87cb-4218-b715-5e60bbac235a', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [12]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery. Here are a few:\n\n1. **Avery Bradley**: An American professional basketball player who plays in the NBA. He has played for several teams, including the Boston Celtics, Detroit Pistons, and Los Angeles Clippers.\n2. **Avery Bradley (musician)**: An American musician and producer who has worked with artists such as Kanye West and Drake.\n3. **Avery Bradley (footballer)**: An American soccer player who has played for several teams, including the Seattle Sounders FC.\n4. **Avery Bradley (author)**: An American author who has written several books, including "The Girl Who Drank the Moon" and "The Vanderbeekers of 141st Street".\n5. **Avery Bradley (TV personality)**: An American TV personality who has appeared on several reality TV shows, including "The Bachelor" and "The Bachelorette".\n6. **Avery Bradley (music)**: Avery Bradley is also a music artist, known for his soulful voice and genre-bending sound.\n\nHowever, wi

## Time to put this together!

In [13]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [14]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])
# spell wrong but still get the right answer because of docs = retriever.invoke(question) enconding and get the closer vectors.

"I couldn't find any information about Averi Lancaster in the provided context. However, I did find information about Avery Lancaster, who is the Co-Founder & Chief Executive Officer (CEO) of Insurellm."

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!